# **Business Customer Support RAG Chatbot**



## 1. Install dependencies

In [1]:
!pip install -q faiss-cpu sentence-transformers openai


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 26.4 MB/s eta 0:00:00


# 2. import labraries

In [2]:
import json
import os
from pathlib import Path
from dotenv import load_dotenv
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from openai import OpenAI
import time

## 3. Load the FAQ knowledge base


In [3]:
BASE_DIR = Path("/content")
KNOWLEDGE_DIR = BASE_DIR / "knowledge"
INDEX_DIR = BASE_DIR / "index"
ENV_DIR = BASE_DIR / "env"

KNOWLEDGE_DIR.mkdir(exist_ok=True)
INDEX_DIR.mkdir(exist_ok=True)
ENV_DIR.mkdir(exist_ok=True)

In [11]:
FAQ_PATH = KNOWLEDGE_DIR / "faq_knowledge_base.json"
with open(FAQ_PATH, "r", encoding="utf-8") as f:
  faqs =  json.load(f)

In [15]:
print("Does the file exist:", FAQ_PATH.exists())
if FAQ_PATH.exists():
  print("File size:", FAQ_PATH.stat().st_size, "bytes")

Does the file exist: True
File size: 30423 bytes


In [16]:
print(f"Saved {len(faqs)} FAQ entries to {FAQ_PATH}")

Saved 100 FAQ entries to /content/knowledge/faq_knowledge_base.json


## 4. Build the FAISS retrieval index

In [17]:
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"

In [18]:
texts = [f"{item['question']} {item['answer']}" for item in faqs]

In [19]:
embedder = SentenceTransformer(EMBEDDING_MODEL_NAME)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [20]:
print(f"Embedding {len(texts)} FAQ entries...")

Embedding 100 FAQ entries...


In [21]:
embeddings = embedder.encode(texts, convert_to_numpy=True, normalize_embeddings=True)

In [22]:
dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings.astype(np.float32))

In [23]:
faiss.write_index(index, str(INDEX_DIR / "faq.index"))
with open(INDEX_DIR / "faq_metadata.json", "w", encoding="utf-8") as f:
    json.dump(faqs, f, ensure_ascii=False, indent=2)

## 5. Set your LLM API key



In [24]:
ENV_FILE = ENV_DIR / ".env"
base_url = "https://api.gapapi.com/v1"
model_name = "gpt-4o"

In [25]:
load_dotenv(ENV_FILE)
api_key = os.getenv("API_KEY")

In [26]:
os.environ["LLM_API_KEY"] = api_key

In [27]:
if api_key:
  print(f'The {ENV_FILE} file exists in the directory.')

The /content/env/.env file exists in the directory.


## 6. Define the chatbot

In [28]:
TOP_K = 3
SIMILARITY_THRESHOLD = 0.45

In [29]:
HANDOFF_MESSAGE = (
    "I'm not fully sure about that based on what I have available. "
    "I don't want to give you inaccurate information, so let me connect you "
    "with a human support agent -- you can reach them at support@example.com "
    "or by typing 'talk to an agent'."
)

In [30]:
class FAQChatbot:
    def __init__(self, faqs, index, embedder):
        self.faqs = faqs
        self.index = index
        self.embedder = embedder
        self.use_llm = bool(os.environ.get("LLM_API_KEY"))
        if self.use_llm:
            self.llm_client = OpenAI(api_key=os.environ["LLM_API_KEY"], base_url=base_url)
            self.model_name = model_name
        self.history = []
    def retrieve(self, query: str, k: int = TOP_K):
        query_vec = self.embedder.encode([query], convert_to_numpy=True, normalize_embeddings=True)
        scores, indices = self.index.search(query_vec.astype(np.float32), k)
        results = []
        for score, idx in zip(scores[0], indices[0]):
            if idx == -1:
                continue
            results.append({"score": float(score), **self.faqs[idx]})
        return results
    def generate_answer(self, user_question: str, retrieved: list) -> str:
        context_block = "\n\n".join(f"Q: {r['question']}\nA: {r['answer']}" for r in retrieved)
        if self.use_llm:
            system_prompt = (
                "You are a friendly, concise customer support assistant. "
                "Answer the user's question using ONLY the information in the "
                "provided FAQ context. If the context doesn't fully answer the "
                "question, say so honestly instead of guessing. Keep answers short."
            )
            user_prompt = f"Customer question: {user_question}\n\nFAQ context:\n{context_block}"
            response = self.llm_client.chat.completions.create(
                model=self.model_name,
                messages=[
                    {"role": "system", "content": system_prompt},
                    *self.history[-6:],
                    {"role": "user", "content": user_prompt},
                ],
                temperature=0.2,
            )
            answer = response.choices[0].message.content
        else:
            answer = retrieved[0]["answer"]
        self.history.append({"role": "user", "content": user_question})
        self.history.append({"role": "assistant", "content": answer})
        return answer
    def ask(self, user_question: str) -> str:
        retrieved = self.retrieve(user_question)
        if not retrieved or retrieved[0]["score"] < SIMILARITY_THRESHOLD:
            self.history.append({"role": "user", "content": user_question})
            self.history.append({"role": "assistant", "content": HANDOFF_MESSAGE})
            return HANDOFF_MESSAGE
        return self.generate_answer(user_question, retrieved)


In [31]:
bot = FAQChatbot(faqs, index, embedder)

## 7. Chat with the bot


In [32]:
name_bot_assist = 'DigiAssist'

In [33]:
sentence = ["Hi there! I'm your customer support assistant.",
       "Ask me about shipping, returns, payments, your account, orders, or warranty.",
       "Type 'exit' to end the chat.\n"]

In [34]:
for sen in sentence:
  time.sleep(5)
  print(f"{name_bot_assist}: {sen}")

while True:
    user_input = input("You: ").strip()
    if user_input.lower() in {"exit", "quit"}:
      time.sleep(5)
      print(f"{name_bot_assist}: Thanks for chatting, have a great day!")
      break
    if not user_input:
        continue
    time.sleep(5)
    answer = bot.ask(user_input)
    print(f"{name_bot_assist}: {answer}\n")


DigiAssist: Hi there! I'm your customer support assistant.
DigiAssist: Ask me about shipping, returns, payments, your account, orders, or warranty.
DigiAssist: Type 'exit' to end the chat.

You: Can I use one account for multiple addresses?
DigiAssist: Yes, you can use one account for multiple addresses. You can save up to 5 addresses in your Address Book under Account Settings and choose which one to use at checkout for each order.

You: Why was my payment declined?
DigiAssist: Your payment was likely declined due to insufficient funds, an expired card, or your bank's fraud protection flagging the transaction. Please verify your card details or contact your bank for more information. No charge is made on a declined payment.

You: Is there free shipping available, and what’s the minimum order?
DigiAssist: Yes, free standard shipping is available for orders over $75 within the country. Orders below this amount incur a flat shipping fee of $5.99.

You: Are there any items that can’t be s